**Table of contents**<a id='toc0_'></a>    
- [What is the set of ambiguous symbols per source?](#toc1_1_)    
    - [Define Functions](#toc1_1_1_)    
    - [ENSG](#toc1_1_2_)    
    - [HGNC](#toc1_1_3_)    
    - [NCBI](#toc1_1_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[What is the set of ambiguous symbols per source?](#toc0_)

Need this to create a better histogram figure. Current one only shows the distribution of ambiguous symbols created by alias-alias collisions by source. I want to be able to illustrate how the ambiguous symbols in each source (created by alias-alias and alias-primary collisions) are shared. 

In [1]:
import pandas as pd
import plotly.express as px
from collections import Counter

### <a id='toc1_1_1_'></a>[Define Functions](#toc0_)

In [2]:
def combine_columns(df, columns_to_combine, columns_to_keep, new_name, columns_to_drop):
    """Combine multiple columns into one while keeping associated data attached.
    Use this function when the columns to combine are easier to list 
    than the columns not to combine.
    
    :param df: The DataFrame containing the columns to be combined
    :param columns_to_combine: List of column names to combine into one
    :param columns_to_keep: List of columns to keep in the final DataFrame
    :param new_name: The name of the new combined column
    :param columns_to_drop: List of columns to drop from the final DataFrame
    :return: A new DataFrame with combined columns and selected columns retained
    """
    og_df = df.copy()

    combined_dfs = []

    # Loop through each column in columns_to_combine and create a new DataFrame
    for col in columns_to_combine:
        temp_df = og_df[list(set([col] + columns_to_keep))].copy()
        temp_df[new_name] = temp_df[col]
        combined_dfs.append(temp_df)

    df_combined = pd.concat(combined_dfs, ignore_index=True)
    df_combined.drop(columns_to_drop, axis=1, inplace=True)
    df_combined.drop_duplicates(inplace=True)
    
    return df_combined

In [3]:
def create_ambiguous_symbol_histogram(XXXX_ambiguous_symbol_counts: pd.DataFrame, source: str, XXXX_ambiguous_symbol_set_count: int):
    """Create a histogram of the frequencies at which aliases are shared

    :param aa_collision_gene_xxxx_df: A df of alias-alias collisions organized by primary gene symbol
    :param source: Representation of the source of the gene records
    :param xxxx_alias_count: Number of aliases total in the source
    :return: A histogram of the percentage of aliases that are shared between 2 genes, 3 genes, and so on
    """

    #Count the frequency at which aliases are shared 
    XXXX_ambiguous_symbol_counts_distribution_df = XXXX_ambiguous_symbol_counts.pivot_table(
    index=["symbol_counts"], aggfunc="size"
    )
    XXXX_ambiguous_symbol_counts_distribution_df = XXXX_ambiguous_symbol_counts_distribution_df.reset_index()
    XXXX_ambiguous_symbol_counts_distribution_df.rename(columns={0: "num_collision_symbol"}, inplace=True)
    XXXX_ambiguous_symbol_counts_distribution_df["percent_collision_symbol"] = (
        XXXX_ambiguous_symbol_counts_distribution_df["num_collision_symbol"] / XXXX_ambiguous_symbol_set_count
    ) * 100

    #Convert to csv
    XXXX_ambiguous_symbol_counts_distribution_df.to_csv(f'../output/ambiguous_symbol_{source}_distribution_df.csv',index=False)

    #Create histogram df 
    
    xxxx_symbol_count_histogram_df = XXXX_ambiguous_symbol_counts_distribution_df.drop(
    "num_collision_symbol", axis=1)

    #Convert to csv
    xxxx_symbol_count_histogram_df.to_csv(f'../output/{source}_ambiguous_symbol_count_histogram_df.csv',index=False)

    return px.bar(xxxx_symbol_count_histogram_df, x="symbol_counts", y="percent_collision_symbol")



### <a id='toc1_1_2_'></a>[ENSG](#toc0_)

import file with ambiguous symbols resulting from alias-alias collisions

In [4]:
merged_alias_aa_collision_ensg_df = pd.read_csv(
    "../output/merged_alias_aa_collision_ensg_df.csv")
merged_alias_aa_collision_ensg_df

,ENSG_ID,collision,primary_gene_symbol,source,NCBI_ID,HGNC_ID,alias_symbol
0,ENSG00000139187,2F1,KLRG1,ENSG,GENE ID:10219,HGNC:6380,"2F1,CLEC15A,MAFA,MAFA-L"
1,ENSG00000005022,2F1,SLC25A5,ENSG,GENE ID:292,HGNC:10991,"2F1,ANT2,T2,T3"
2,ENSG00000163220,60B8AG,S100A9,ENSG,GENE ID:6280,HGNC:10499,"60B8AG,CAGB,CFAG,CGLB,LIAG,MAC387,MIF,MRP-14,M..."
3,ENSG00000143546,60B8AG,S100A8,ENSG,GENE ID:6279,HGNC:10498,"60B8AG,CAGA,CFAG,CGLA,MRP-8,MRP8,P8,S100-A8"
4,ENSG00000206832,87U6,RNU6V,ENSG,NaN,HGNC:10230,"87U6,LH87"
...,...,...,...,...,...,...,...
4727,ENSG00000164756,ZNT8,SLC30A8,ENSG,GENE ID:169026,HGNC:20303,"ZNT-8,ZNT8"
4728,ENSG00000170385,ZRC1,SLC30A1,ENSG,GENE ID:7779,HGNC:11012,"ZNT1,ZRC1"
4729,ENSG00000196660,ZRC1,SLC30A10,ENSG,GENE ID:55532,HGNC:25355,"DKFZP547M236,ZNT-10,ZNT10,ZNT8,ZRC1"
4730,ENSG00000203995,ZYG11,ZYG11A,ENSG,GENE ID:440590,HGNC:32058,ZYG11


In [5]:
aa_collision_alias_ensg_set = set(
    merged_alias_aa_collision_ensg_df["collision"]
)

In [6]:
l_aa_collision_alias_ensg_set = set(s.lower() for s in aa_collision_alias_ensg_set if isinstance(s, str))

In [7]:
len(l_aa_collision_alias_ensg_set)

1615

import file with ambiguous symbols resulting from alias-primary collisions

In [8]:
ap_collision_ensg_df = pd.read_csv(
    "../output/merged_alias_ap_collision_ensg_df.csv")
ap_collision_ensg_df

,ENSG_ID,primary_gene_symbol,collision,source,NCBI_ID,HGNC_ID,alias_symbol
0,ENSG00000283293,RN7SK,7SK,ENSG,GENE ID:125050,HGNC:10037,7SK
1,ENSG00000057252,SOAT1,ACAT1,ENSG,GENE ID:6646,HGNC:11177,"ACAT,ACAT1,SOAT,STAT"
2,ENSG00000167780,SOAT2,ACAT2,ENSG,GENE ID:8435,HGNC:11178,ACAT2
3,ENSG00000004779,NDUFAB1,ACP1,ENSG,GENE ID:4706,HGNC:7694,"ACP,ACP1,FASN2A,SDAP"
4,ENSG00000220267,ACTBP8,ACTBP2,ENSG,NaN,HGNC:141,ACTBP2
...,...,...,...,...,...,...,...
762,ENSG00000197961,ZNF121,ZNF20,ENSG,GENE ID:7675,HGNC:12904,"D19S204,ZHC32,ZNF20"
763,ENSG00000110315,RNF141,ZNF230,ENSG,GENE ID:50862,HGNC:21159,"ZFP26,ZNF230"
764,ENSG00000188801,ZNF322P1,ZNF322,ENSG,NaN,HGNC:14003,"ZNF322,ZNF322B"
765,ENSG00000103994,ZNF106,ZNF474,ENSG,GENE ID:64397,HGNC:12886,"SH3BP3,ZFP106,ZNF474"


In [9]:
ap_collision_ensg_df.loc[ap_collision_ensg_df["collision"] == "SKI"]

,ENSG_ID,primary_gene_symbol,collision,source,NCBI_ID,HGNC_ID,alias_symbol
635,ENSG00000054392,HHAT,SKI,ENSG,GENE ID:55733,HGNC:18270,"FLJ10724,GUP2,MART-2,MART2,RASP,SIT,SKI,SKN"
636,ENSG00000280680,HHAT,SKI,ENSG,GENE ID:55733,HGNC:18270,"FLJ10724,GUP2,MART-2,MART2,RASP,SIT,SKI,SKN"


In [10]:
ap_collision_alias_ensg_set = set(
    ap_collision_ensg_df["collision"]
)

l_ap_collision_alias_ensg_set = set(s.lower() for s in ap_collision_alias_ensg_set if isinstance(s, str))


len(l_ap_collision_alias_ensg_set)

606

combine sets to create set of ambiguous symbols for ENSG

In [11]:
ensg_ambiguous_symbol_set = l_aa_collision_alias_ensg_set.union(
    l_ap_collision_alias_ensg_set
)
ensg_ambiguous_symbol_set_count = len(ensg_ambiguous_symbol_set)
ensg_ambiguous_symbol_set_count

2163

In [12]:
both_collision_ensg_set = l_aa_collision_alias_ensg_set & l_ap_collision_alias_ensg_set
len(both_collision_ensg_set)

58

In [13]:
only_aa_collision_alias_ensg_set = l_aa_collision_alias_ensg_set - both_collision_ensg_set
len(only_aa_collision_alias_ensg_set)

1557

In [14]:
only_ap_collision_alias_ensg_set = l_ap_collision_alias_ensg_set - both_collision_ensg_set
len(only_ap_collision_alias_ensg_set)

548

In [15]:
subset_genes_ensg_df = pd.read_csv(
        ("../output/subset_genes_ensg_df.csv"),dtype={"NCBI_ID": str,"HGNC_ID":str})

In [16]:
ensg_df = subset_genes_ensg_df.copy()
ensg_df['primary_gene_symbol_lower'] = ensg_df['primary_gene_symbol'].str.lower()
ensg_df.drop(["primary_gene_symbol"], axis=1, inplace=True)
ensg_df['alias_symbol'] = ensg_df['alias_symbol'].str.lower()
ensg_df = ensg_df.drop_duplicates(subset=['primary_gene_symbol_lower', 'alias_symbol'], keep='first')
ensg_df

,ENSG_ID,NCBI_ID,HGNC_ID,alias_symbol,primary_gene_symbol_lower
0,ENSG00000210049,NaN,HGNC:7481,mttf,mt-tf
1,ENSG00000210049,NaN,HGNC:7481,trnf,mt-tf
2,ENSG00000211459,NaN,HGNC:7470,12s,mt-rnr1
3,ENSG00000211459,NaN,HGNC:7470,mots-c,mt-rnr1
4,ENSG00000211459,NaN,HGNC:7470,mtrnr1,mt-rnr1
...,...,...,...,...,...
95019,ENSG00000197989,GENE ID:85028,HGNC:30062,linc00100,snhg12
95020,ENSG00000197989,GENE ID:85028,HGNC:30062,pnas-123,snhg12
95021,ENSG00000229388,NaN,HGNC:52502,linc01715,taf12-dt
95022,ENSG00000274978,GENE ID:26824,HGNC:10108,rnu11-1,rnu11


make a general symbol column with primary and alias symbols

In [17]:
ensg_df = combine_columns(ensg_df, ["primary_gene_symbol_lower", "alias_symbol"], ["primary_gene_symbol_lower", "ENSG_ID", "HGNC_ID","NCBI_ID"], "symbol", "alias_symbol")
ensg_df

,ENSG_ID,HGNC_ID,primary_gene_symbol_lower,NCBI_ID,symbol
0,ENSG00000210049,HGNC:7481,mt-tf,NaN,mt-tf
2,ENSG00000211459,HGNC:7470,mt-rnr1,NaN,mt-rnr1
5,ENSG00000210077,HGNC:7500,mt-tv,NaN,mt-tv
7,ENSG00000210082,HGNC:7471,mt-rnr2,NaN,mt-rnr2
10,ENSG00000209082,HGNC:7490,mt-tl1,NaN,mt-tl1
...,...,...,...,...,...
147837,ENSG00000197989,HGNC:30062,snhg12,GENE ID:85028,linc00100
147838,ENSG00000197989,HGNC:30062,snhg12,GENE ID:85028,pnas-123
147839,ENSG00000229388,HGNC:52502,taf12-dt,NaN,linc01715
147840,ENSG00000274978,HGNC:10108,rnu11,GENE ID:26824,rnu11-1


use list of ambiguous symbols and ensg_combined_concept_ids_df to create a histogram of how many genes share ambiguous symbols

In [18]:
ensg_symbol_value_counts = ensg_df["symbol"].value_counts()

ensg_ambiguous_symbol_counts = ensg_symbol_value_counts[ensg_symbol_value_counts.index.isin(ensg_ambiguous_symbol_set)]
ensg_ambiguous_symbol_counts = ensg_ambiguous_symbol_counts.reset_index()
ensg_ambiguous_symbol_counts.columns = ["symbols", "symbol_counts"]
ensg_ambiguous_symbol_counts

,symbols,symbol_counts
0,mt1,11
1,hox1,10
2,p40,10
3,hox2,9
4,p14,9
...,...,...
2158,cdk4i,2
2159,g6pt,2
2160,l30,2
2161,prp,2


In [19]:
len(set(ensg_ambiguous_symbol_counts["symbols"]))

2163

In [20]:
ensg_ambiguous_symbol_set_count

2163

In [21]:
create_ambiguous_symbol_histogram(ensg_ambiguous_symbol_counts, "ENSG", ensg_ambiguous_symbol_set_count)

made in the create_ambiguous_symbol_histogram function

In [22]:
ensg_ambiguous_symbol_counts_distribution_df = pd.read_csv(
    "../output/ambiguous_symbol_ensg_distribution_df.csv")
ensg_ambiguous_symbol_counts_distribution_df

,symbol_counts,num_collision_symbol,percent_collision_symbol
0,2,1837,84.928340
1,3,237,10.957004
2,4,49,2.265372
3,5,16,0.739713
4,6,13,0.601017
5,7,4,0.184928
6,8,2,0.092464
7,9,2,0.092464
8,10,2,0.092464
9,11,1,0.046232


In [23]:
ensg_ambiguous_symbol_counts_distribution_df["percent_collision_symbol"].sum()

100.0

In [24]:
# Filter out rows with symbol_counts < 6
less_than_six_ensg_ambiguous_symbol_counts_distribution_df = ensg_ambiguous_symbol_counts_distribution_df[ensg_ambiguous_symbol_counts_distribution_df["symbol_counts"] < 6].copy()

# Group all rows where symbol_counts >= 6
equal_more_than_six_ensg_ambiguous_symbol_counts_distribution_df = ensg_ambiguous_symbol_counts_distribution_df[ensg_ambiguous_symbol_counts_distribution_df["symbol_counts"] >= 6].agg({
    'num_collision_symbol': 'sum',
    'percent_collision_symbol': 'sum'
})

# Create a new row for '6+'
agg_row = pd.DataFrame({
    'symbol_counts': ['6+'],
    'num_collision_symbol': [equal_more_than_six_ensg_ambiguous_symbol_counts_distribution_df['num_collision_symbol']],
    'percent_collision_symbol': [equal_more_than_six_ensg_ambiguous_symbol_counts_distribution_df['percent_collision_symbol']]
})

# Concatenate and reset index
limited_ensg_ambiguous_symbol_counts_distribution_df = pd.concat([less_than_six_ensg_ambiguous_symbol_counts_distribution_df, agg_row], ignore_index=True)
limited_ensg_ambiguous_symbol_counts_distribution_df

,symbol_counts,num_collision_symbol,percent_collision_symbol
0,2,1837.0,84.928340
1,3,237.0,10.957004
2,4,49.0,2.265372
3,5,16.0,0.739713
4,6+,24.0,1.109570


### <a id='toc1_1_3_'></a>[HGNC](#toc0_)

import file with ambiguous symbols resulting from alias-alias collisions

In [25]:
merged_alias_aa_collision_hgnc_df = pd.read_csv(
    "../output/merged_alias_aa_collision_hgnc_df.csv")
merged_alias_aa_collision_hgnc_df

,HGNC_ID,primary_gene_symbol,collision,source,NCBI_ID,ENSG_ID,alias_symbol
0,HGNC:10991,SLC25A5,2F1,HGNC,GENE ID:292,ENSG00000005022,"2F1,T2,T3"
1,HGNC:6380,KLRG1,2F1,HGNC,GENE ID:10219,ENSG00000139187,"2F1,CLEC15A,MAFA,MAFA-L"
2,HGNC:10499,S100A9,60B8AG,HGNC,GENE ID:6280,ENSG00000163220,"60B8AG,CGLB,LIAG,MAC387,MIF,MRP-14,MRP14,NIF,P..."
3,HGNC:10498,S100A8,60B8AG,HGNC,GENE ID:6279,ENSG00000143546,"60B8AG,CGLA,MRP-8,MRP8,P8,S100-A8"
4,HGNC:10230,RNU6V,87U6,HGNC,GENE ID:6071,ENSG00000206832,"87U6,LH87"
...,...,...,...,...,...,...,...
2817,HGNC:28393,SPATA2L,tamo,HGNC,GENE ID:124044,ENSG00000158792,"MGC26885,tamo"
2818,HGNC:1132,BTG3,tob55,HGNC,GENE ID:10950,ENSG00000154640,"ANA,ANA/BTG3,APRO4,tob55"
2819,HGNC:3818,FOXM1,trident,HGNC,GENE ID:2305,ENSG00000111206,"HFH-11,HNF-3,INS-1,MPHOSPH2,MPP2,TGT3,trident"
2820,HGNC:7494,MT-TP,trnP,HGNC,GENE ID:4571,ENSG00000210196,trnP


In [26]:
aa_collision_alias_hgnc_set = set(
    merged_alias_aa_collision_hgnc_df["collision"]
)

In [27]:
l_aa_collision_alias_hgnc_set = set(s.lower() for s in aa_collision_alias_hgnc_set if isinstance(s, str))

In [28]:
len(l_aa_collision_alias_hgnc_set)

1230

import file with ambiguous symbols resulting from alias-primary collisions

In [29]:
ap_collision_hgnc_df = pd.read_csv(
    "../output/merged_alias_ap_collision_hgnc_df.csv")
ap_collision_hgnc_df

,HGNC_ID,primary_gene_symbol,collision,source,NCBI_ID,ENSG_ID,alias_symbol
0,HGNC:14947,PPP1R12C,AAVS1,HGNC,GENE ID:54776,ENSG00000125503,"AAVS1,DKFZP434D0412,MBS85,p84,p85"
1,HGNC:11177,SOAT1,ACAT1,HGNC,GENE ID:6646,ENSG00000057252,"ACAT,ACAT1"
2,HGNC:11178,SOAT2,ACAT2,HGNC,GENE ID:8435,ENSG00000167780,ACAT2
3,HGNC:1612,CCRL2,ACKR5,HGNC,GENE ID:9034,ENSG00000121797,"ACKR5,CKRX,CRAM-A,CRAM-B,HCR"
4,HGNC:7694,NDUFAB1,ACP1,HGNC,GENE ID:4706,ENSG00000004779,"ACP,ACP1,FASN2A,SDAP"
...,...,...,...,...,...,...,...
541,HGNC:3147,ECEL1,XCE,HGNC,GENE ID:9427,ENSG00000171551,"DINE,XCE"
542,HGNC:12904,ZNF121,ZNF20,HGNC,GENE ID:7675,ENSG00000197961,"ZHC32,ZNF20"
543,HGNC:21159,RNF141,ZNF230,HGNC,GENE ID:50862,ENSG00000110315,"ZFP26,ZNF230"
544,HGNC:12886,ZNF106,ZNF474,HGNC,GENE ID:64397,ENSG00000103994,"SH3BP3,ZNF474"


In [30]:
ap_collision_alias_hgnc_set = set(
    ap_collision_hgnc_df["collision"]
)

l_ap_collision_alias_hgnc_set = set(s.lower() for s in ap_collision_alias_hgnc_set if isinstance(s, str))

len(l_ap_collision_alias_hgnc_set)

492

combine lists without duplicates to create list of ambiguous symbols for HGNC

In [31]:
hgnc_ambiguous_symbol_set = l_aa_collision_alias_hgnc_set.union(
    l_ap_collision_alias_hgnc_set
)
hgnc_ambiguous_symbol_set_count = len(hgnc_ambiguous_symbol_set)
hgnc_ambiguous_symbol_set_count

1673

In [32]:
both_collision_hgnc_set = l_aa_collision_alias_hgnc_set & l_ap_collision_alias_hgnc_set
len(both_collision_hgnc_set)

49

In [33]:
only_aa_collision_alias_hgnc_set = l_aa_collision_alias_hgnc_set - both_collision_hgnc_set
len(only_aa_collision_alias_hgnc_set)

1181

In [34]:
only_ap_collision_alias_hgnc_set = l_ap_collision_alias_hgnc_set - both_collision_hgnc_set
len(only_ap_collision_alias_hgnc_set)

443

In [35]:
subset_genes_hgnc_df = pd.read_csv(
        ("../output/subset_genes_hgnc_df.csv"),dtype={"NCBI_ID": str,"HGNC_ID":str})

In [36]:
hgnc_df = subset_genes_hgnc_df.copy()
hgnc_df['primary_gene_symbol_lower'] = hgnc_df['primary_gene_symbol'].str.lower()
hgnc_df.drop(["primary_gene_symbol"], axis=1, inplace=True)
hgnc_df['alias_symbol'] = hgnc_df['alias_symbol'].str.lower()
hgnc_df = hgnc_df.drop_duplicates(subset=['primary_gene_symbol_lower', 'alias_symbol'], keep='first')
hgnc_df

,HGNC_ID,alias_symbol,NCBI_ID,ENSG_ID,primary_gene_symbol_lower
0,HGNC:100,bnac2,GENE ID:41,ENSG00000110881,asic1
1,HGNC:100,hbnac2,GENE ID:41,ENSG00000110881,asic1
2,HGNC:10000,NaN,GENE ID:5999,ENSG00000117152,rgs4
3,HGNC:10001,NaN,GENE ID:8490,ENSG00000143248,rgs5
4,HGNC:10002,NaN,GENE ID:9628,ENSG00000182732,rgs6
...,...,...,...,...,...
66194,HGNC:9997,rgs-r,GENE ID:6004,ENSG00000143333,rgs16
66195,HGNC:9998,NaN,GENE ID:5997,ENSG00000116741,rgs2
66196,HGNC:9999,c2pa,GENE ID:5998,ENSG00000138835,rgs3
66197,HGNC:9999,flj20370,GENE ID:5998,ENSG00000138835,rgs3


make a general column with primary and alias symbols

In [37]:
hgnc_df = combine_columns(hgnc_df, ["primary_gene_symbol_lower", "alias_symbol"], ["primary_gene_symbol_lower", "ENSG_ID", "HGNC_ID","NCBI_ID"], "symbol", "alias_symbol")
hgnc_df

,ENSG_ID,HGNC_ID,primary_gene_symbol_lower,NCBI_ID,symbol
0,ENSG00000110881,HGNC:100,asic1,GENE ID:41,asic1
2,ENSG00000117152,HGNC:10000,rgs4,GENE ID:5999,rgs4
3,ENSG00000143248,HGNC:10001,rgs5,GENE ID:8490,rgs5
4,ENSG00000182732,HGNC:10002,rgs6,GENE ID:9628,rgs6
5,ENSG00000182901,HGNC:10003,rgs7,GENE ID:6000,rgs7
...,...,...,...,...,...
132325,ENSG00000143333,HGNC:9997,rgs16,GENE ID:6004,rgs-r
132326,ENSG00000116741,HGNC:9998,rgs2,GENE ID:5997,NaN
132327,ENSG00000138835,HGNC:9999,rgs3,GENE ID:5998,c2pa
132328,ENSG00000138835,HGNC:9999,rgs3,GENE ID:5998,flj20370


use list of ambiguous symbols and hgnc_combined_concept_ids_df to create a histogram of how many genes share ambiguous symbols

In [38]:
hgnc_symbol_value_counts = hgnc_df["symbol"].value_counts()

hgnc_ambiguous_symbol_counts = hgnc_symbol_value_counts[hgnc_symbol_value_counts.index.isin(hgnc_ambiguous_symbol_set)]
hgnc_ambiguous_symbol_counts = hgnc_ambiguous_symbol_counts.reset_index()
hgnc_ambiguous_symbol_counts.columns = ["symbols", "symbol_counts"]
hgnc_ambiguous_symbol_counts

,symbols,symbol_counts
0,p40,10
1,p14,9
2,p18,7
3,pap,7
4,asp,7
...,...,...
1668,sin,2
1669,rho,2
1670,mio,2
1671,las1,2


In [39]:
len(set(hgnc_ambiguous_symbol_counts["symbols"]))

1673

In [40]:
hgnc_ambiguous_symbol_set_count

1673

In [41]:
create_ambiguous_symbol_histogram(hgnc_ambiguous_symbol_counts, "HGNC", hgnc_ambiguous_symbol_set_count)

In [42]:
hgnc_ambiguous_symbol_counts_distribution_df = pd.read_csv(
    "../output/ambiguous_symbol_hgnc_distribution_df.csv")
hgnc_ambiguous_symbol_counts_distribution_df

,symbol_counts,num_collision_symbol,percent_collision_symbol
0,2,1400,83.682008
1,3,197,11.775254
2,4,43,2.570233
3,5,18,1.075912
4,6,9,0.537956
5,7,4,0.239091
6,9,1,0.059773
7,10,1,0.059773


In [43]:
hgnc_ambiguous_symbol_counts_distribution_df["percent_collision_symbol"].sum()

100.0

In [44]:
# Filter out rows with symbol_counts < 6
less_than_six_hgnc_ambiguous_symbol_counts_distribution_df = hgnc_ambiguous_symbol_counts_distribution_df[hgnc_ambiguous_symbol_counts_distribution_df["symbol_counts"] < 6].copy()

# Group all rows where symbol_counts >= 6
equal_more_than_six_hgnc_ambiguous_symbol_counts_distribution_df = hgnc_ambiguous_symbol_counts_distribution_df[hgnc_ambiguous_symbol_counts_distribution_df["symbol_counts"] >= 6].agg({
    'num_collision_symbol': 'sum',
    'percent_collision_symbol': 'sum'
})

# Create a new row for '6+'
agg_row = pd.DataFrame({
    'symbol_counts': ['6+'],
    'num_collision_symbol': [equal_more_than_six_hgnc_ambiguous_symbol_counts_distribution_df['num_collision_symbol']],
    'percent_collision_symbol': [equal_more_than_six_hgnc_ambiguous_symbol_counts_distribution_df['percent_collision_symbol']]
})

# Concatenate and reset index
limited_hgnc_ambiguous_symbol_counts_distribution_df = pd.concat([less_than_six_hgnc_ambiguous_symbol_counts_distribution_df, agg_row], ignore_index=True)
limited_hgnc_ambiguous_symbol_counts_distribution_df

,symbol_counts,num_collision_symbol,percent_collision_symbol
0,2,1400.0,83.682008
1,3,197.0,11.775254
2,4,43.0,2.570233
3,5,18.0,1.075912
4,6+,15.0,0.896593


### <a id='toc1_1_4_'></a>[NCBI](#toc0_)

import file with ambiguous symbols resulting from alias-alias collisions

In [45]:
merged_alias_aa_collision_ncbi_df = pd.read_csv(
    "../output/merged_alias_aa_collision_ncbi_df.csv")
merged_alias_aa_collision_ncbi_df

,NCBI_ID,primary_gene_symbol,collision,gene_type,source,HGNC_ID,ENSG_ID,alias_symbol
0,GENE ID:657,BMPR1A,10q23del,protein-coding,NCBI,HGNC:1076,ENSG00000107779,"10q23del,ACVRLK3,ALK-3,ALK3,BMPR-1A,CD292,SKR5"
1,GENE ID:5728,PTEN,10q23del,protein-coding,NCBI,HGNC:9588,ENSG00000171862,"10q23del,BZS,CWS1,DEC,GLM2,MHAM,MMAC1,PTEN1,PT..."
2,GENE ID:239,ALOX12,12-LOX,protein-coding,NCBI,HGNC:429,ENSG00000108839,"12-LOX,12S-LOX,LOG12"
3,GENE ID:246,ALOX15,12-LOX,protein-coding,NCBI,HGNC:433,ENSG00000161905,"12-LOX,15-LOX,15-LOX-1,LOG15"
4,GENE ID:1645,AKR1C1,20-ALPHA-HSD,protein-coding,NCBI,HGNC:384,ENSG00000187134,"2-ALPHA-HSD,20-ALPHA-HSD,C9,DD1,DD1/DD2,DDH,DD..."
...,...,...,...,...,...,...,...,...
8792,GENE ID:100874116,EGFLAM-AS3,tbp1,ncRNA,NCBI,HGNC:41167,NaN,tbp1
8793,GENE ID:80222,TARS2,thrRS,protein-coding,NCBI,HGNC:30740,ENSG00000143374,"COXPD21,TARSL1,thrRS"
8794,GENE ID:3921,RPSA,uS2,protein-coding,NCBI,HGNC:6502,ENSG00000168028,"37LRP,67LR,ICAS,LAMBR,LAMR1,LBP,LBP/p40,LRP,LR..."
8795,GENE ID:9113,LATS1,wts,protein-coding,NCBI,HGNC:6514,ENSG00000131023,"WARTS,wts"


In [46]:
aa_collision_alias_ncbi_set = set(
    merged_alias_aa_collision_ncbi_df["collision"]
)

In [47]:
l_aa_collision_alias_ncbi_set = set(s.lower() for s in aa_collision_alias_ncbi_set if isinstance(s, str))

In [48]:
len(l_aa_collision_alias_ncbi_set)

3648

import file with ambiguous symbols resulting from alias-primary collisions

In [49]:
ap_collision_ncbi_df = pd.read_csv(
    "../output/merged_alias_ap_collision_ncbi_df.csv")
ap_collision_ncbi_df

,NCBI_ID,primary_gene_symbol,gene_type,collision,source,HGNC_ID,ENSG_ID,alias_symbol
0,GENE ID:3494,IGHA2,other,A2M,NCBI,HGNC:5479,ENSG00000211890,A2M
1,GENE ID:54776,PPP1R12C,protein-coding,AAVS1,NCBI,HGNC:14947,ENSG00000125503,"AAVS1,LENG3,MBS85,p84,p85"
2,GENE ID:6646,SOAT1,protein-coding,ACAT1,NCBI,HGNC:11177,ENSG00000057252,"ACACT,ACAT,ACAT-1,ACAT1,SOAT,STAT"
3,GENE ID:8435,SOAT2,protein-coding,ACAT2,NCBI,HGNC:11178,ENSG00000167780,"ACACT2,ACAT2,ARGP2"
4,GENE ID:9034,CCRL2,protein-coding,ACKR5,NCBI,HGNC:1612,ENSG00000121797,"ACKR5,CKRX,CRAM,CRAM-A,CRAM-B,HCR"
...,...,...,...,...,...,...,...,...
1608,GENE ID:29903,CCDC106,protein-coding,ZNF581,NCBI,HGNC:30181,ENSG00000173581,"HSU79303,ZNF581"
1609,GENE ID:146540,ZNF785,protein-coding,ZNF688,NCBI,HGNC:26496,ENSG00000197162,ZNF688
1610,GENE ID:57829,ZP4,protein-coding,ZP1,NCBI,HGNC:15770,ENSG00000116996,"ZBP,ZP1,ZP1B,ZPB,ZPB2,Zp-4"
1611,GENE ID:55663,ZNF446,protein-coding,ZSCAN30,NCBI,HGNC:21036,ENSG00000083838,"ZKSCAN20,ZSCAN30,ZSCAN52"


In [50]:
ap_collision_alias_ncbi_set = set(
    ap_collision_ncbi_df["collision"]
)

l_ap_collision_alias_ncbi_set = set(s.lower() for s in ap_collision_alias_ncbi_set if isinstance(s, str))

len(l_ap_collision_alias_ncbi_set)

1394

combine lists without duplicates to create list of ambiguous symbols for NCBI

In [51]:
ncbi_ambiguous_symbol_set = l_aa_collision_alias_ncbi_set.union(
    l_ap_collision_alias_ncbi_set
)
ncbi_ambiguous_symbol_set_count = len(ncbi_ambiguous_symbol_set)
ncbi_ambiguous_symbol_set_count

4872

In [52]:
both_collision_ncbi_set = l_aa_collision_alias_ncbi_set & l_ap_collision_alias_ncbi_set
len(both_collision_ncbi_set)

170

In [53]:
only_aa_collision_alias_ncbi_set = l_aa_collision_alias_ncbi_set - both_collision_ncbi_set
len(only_aa_collision_alias_ncbi_set)

3478

In [54]:
only_ap_collision_alias_ncbi_set = l_ap_collision_alias_ncbi_set - both_collision_ncbi_set
len(only_ap_collision_alias_ncbi_set)

1224

In [55]:
subset_genes_ncbi_df = pd.read_csv(
        ("../output/subset_genes_ncbi_df.csv"),dtype={"NCBI_ID": str,"HGNC_ID":str})

In [56]:
ncbi_df = subset_genes_ncbi_df.copy()
ncbi_df['primary_gene_symbol_lower'] = ncbi_df['primary_gene_symbol'].str.lower()
ncbi_df.drop(["primary_gene_symbol"], axis=1, inplace=True)
ncbi_df['alias_symbol'] = ncbi_df['alias_symbol'].str.lower()
ncbi_df = ncbi_df.drop_duplicates(subset=['primary_gene_symbol_lower', 'alias_symbol'], keep='first')
ncbi_df

,NCBI_ID,alias_symbol,gene_type,HGNC_ID,ENSG_ID,primary_gene_symbol_lower
0,GENE ID:1,a1b,protein-coding,HGNC:5,ENSG00000121410,a1bg
1,GENE ID:1,abg,protein-coding,HGNC:5,ENSG00000121410,a1bg
2,GENE ID:1,gab,protein-coding,HGNC:5,ENSG00000121410,a1bg
3,GENE ID:1,hyst2477,protein-coding,HGNC:5,ENSG00000121410,a1bg
4,GENE ID:2,a2md,protein-coding,HGNC:7,ENSG00000175899,a2m
...,...,...,...,...,...,...
91350,GENE ID:141732005,NaN,ncRNA,HGNC:40064,NaN,adcy2-as1
91351,GENE ID:141732006,NaN,ncRNA,HGNC:41074,NaN,nsg2-as1
91352,GENE ID:141732007,NaN,ncRNA,HGNC:58430,NaN,st18-as1
91353,GENE ID:141732008,NaN,ncRNA,HGNC:58437,NaN,mical2-as1


make a general symbol column with primary and alias symbols

In [57]:
ncbi_df = combine_columns(ncbi_df, ["primary_gene_symbol_lower", "alias_symbol"], ["primary_gene_symbol_lower", "ENSG_ID", "HGNC_ID","NCBI_ID"], "symbol", "alias_symbol")
ncbi_df

,ENSG_ID,HGNC_ID,primary_gene_symbol_lower,NCBI_ID,symbol
0,ENSG00000121410,HGNC:5,a1bg,GENE ID:1,a1bg
4,ENSG00000175899,HGNC:7,a2m,GENE ID:2,a2m
8,ENSG00000171428,HGNC:7645,nat1,GENE ID:9,nat1
12,ENSG00000156006,HGNC:7646,nat2,GENE ID:10,nat2
15,NaN,HGNC:15,natp,GENE ID:11,natp
...,...,...,...,...,...
182509,NaN,HGNC:40064,adcy2-as1,GENE ID:141732005,NaN
182510,NaN,HGNC:41074,nsg2-as1,GENE ID:141732006,NaN
182511,NaN,HGNC:58430,st18-as1,GENE ID:141732007,NaN
182512,NaN,HGNC:58437,mical2-as1,GENE ID:141732008,NaN


use list of ambiguous symbols and ncbi_combined_concept_ids_df to create a histogram of how many genes share ambiguous symbols

In [58]:
ncbi_symbol_value_counts = ncbi_df["symbol"].value_counts()

ncbi_ambiguous_symbol_counts = ncbi_symbol_value_counts[ncbi_symbol_value_counts.index.isin(ncbi_ambiguous_symbol_set)]
ncbi_ambiguous_symbol_counts = ncbi_ambiguous_symbol_counts.reset_index()
ncbi_ambiguous_symbol_counts.columns = ["symbols", "symbol_counts"]
ncbi_ambiguous_symbol_counts

,symbols,symbol_counts
0,vh,37
1,h4c1,14
2,h4c3,14
3,h4c8,14
4,h4c12,14
...,...,...
4867,acecs1,2
4868,try-gta5-1,2
4869,ia4,2
4870,trp-tgg3-4,2


In [59]:
create_ambiguous_symbol_histogram(ncbi_ambiguous_symbol_counts, "NCBI", ncbi_ambiguous_symbol_set_count)

In [60]:
ncbi_ambiguous_symbol_counts_distribution_df = pd.read_csv(
    "../output/ambiguous_symbol_ncbi_distribution_df.csv")
ncbi_ambiguous_symbol_counts_distribution_df

,symbol_counts,num_collision_symbol,percent_collision_symbol
0,2,4009,82.286535
1,3,551,11.309524
2,4,145,2.976190
3,5,71,1.457307
4,6,32,0.656814
5,7,18,0.369458
6,8,8,0.164204
7,9,5,0.102627
8,10,12,0.246305
9,11,3,0.061576


In [61]:
ncbi_ambiguous_symbol_counts_distribution_df["percent_collision_symbol"].sum()

100.0

In [62]:
# Filter out rows with symbol_counts < 6
less_than_six_ncbi_ambiguous_symbol_counts_distribution_df = ncbi_ambiguous_symbol_counts_distribution_df[ncbi_ambiguous_symbol_counts_distribution_df["symbol_counts"] < 6].copy()

# Group all rows where symbol_counts >= 6
equal_more_than_six_ncbi_ambiguous_symbol_counts_distribution_df = ncbi_ambiguous_symbol_counts_distribution_df[ncbi_ambiguous_symbol_counts_distribution_df["symbol_counts"] >= 6].agg({
    'num_collision_symbol': 'sum',
    'percent_collision_symbol': 'sum'
})

# Create a new row for '6+'
agg_row = pd.DataFrame({
    'symbol_counts': ['6+'],
    'num_collision_symbol': [equal_more_than_six_ncbi_ambiguous_symbol_counts_distribution_df['num_collision_symbol']],
    'percent_collision_symbol': [equal_more_than_six_ncbi_ambiguous_symbol_counts_distribution_df['percent_collision_symbol']]
})

# Concatenate and reset index
limited_ncbi_ambiguous_symbol_counts_distribution_df = pd.concat([less_than_six_ncbi_ambiguous_symbol_counts_distribution_df, agg_row], ignore_index=True)
limited_ncbi_ambiguous_symbol_counts_distribution_df

,symbol_counts,num_collision_symbol,percent_collision_symbol
0,2,4009.0,82.286535
1,3,551.0,11.309524
2,4,145.0,2.976190
3,5,71.0,1.457307
4,6+,96.0,1.970443


In [63]:
ncbi_ensg_ambiguous_symbol_set = ncbi_ambiguous_symbol_set.union(
    ensg_ambiguous_symbol_set
)

In [64]:
ncbi_ensg_hgnc_ambiguous_symbol_set = ncbi_ensg_ambiguous_symbol_set.union(
    hgnc_ambiguous_symbol_set
)

In [65]:
ambiguous_symbol_set = ncbi_ensg_hgnc_ambiguous_symbol_set
len(ambiguous_symbol_set)

5009